In [14]:
from pyomo.environ import *

m = ConcreteModel()
m.H = Set(initialize=['Bryan', 'Conroe','Hempstead','Montgomery','Waco']) #Suppliers/warehouse
m.I = Set(initialize=['Austin', 'Dallas','Houston']) #Markets

# Define parameters for cost, supply, and demand
#First set of distances (h,i)
shipcost = {('Bryan','Austin'):16, ('Bryan','Dallas'):27, ('Bryan','Houston'):15,
            ('Conroe','Austin'):27, ('Conroe','Dallas'):30, ('Conroe','Houston'):6,
            ('Hempstead','Austin'):15, ('Hempstead','Dallas'):28, ('Hempstead','Houston'):9,
            ('Montgomery','Austin'):21, ('Montgomery','Dallas'):31, ('Montgomery','Houston'):10,
            ('Waco','Austin'):16, ('Waco','Dallas'):15, ('Waco','Houston'):29,
            }
#Second set of demands
demand = {('Austin'): 5000,
          ('Dallas'): 6000,
          ('Houston'): 7000,
            }
capacity = {('Bryan'): 5000,
          ('Conroe'): 4000,
          ('Hempstead'): 4000,
          ('Montgomery'): 3000,
          ('Waco'): 5000,
            }
rent = {('Bryan'): 50000,
        ('Conroe'): 30000,
        ('Hempstead'): 20000,
        ('Montgomery'): 30000,
        ('Waco'): 60000,
         }
m.x1 = Var(m.H, m.I, domain = NonNegativeReals)
m.a = Var(domain = NonNegativeReals)
m.b = Var(domain = NonNegativeReals)
m.y1 = Var(m.H, domain = Binary)

#Prices rules
def cost_1(m):
    return m.a == sum(shipcost[h,i]* m.x1[h,i] for h in m.H for i in m.I)
m.con1 = Constraint(rule = cost_1)

def cost_2(m):
    return m.b == sum(rent[h]*m.y1[h] for h in m.H) #Rent activates if the respective binary is 1
m.con2 = Constraint(rule = cost_2)

def objective_rule(m):
    return m.a+m.b
m.total_cost = Objective(rule = objective_rule, sense=minimize)

def demand_const(m,i):
    return sum(m.x1[h,i] for h in m.H) >= demand[i]
m.con3 = Constraint(m.I, rule = demand_const)
#
def capacity_const(m,h):
    return sum(m.x1[h,i] for i in m.I) <= capacity[h]
m.con4 = Constraint(m.H, rule = capacity_const)

def capacity_const(m,h):
    return sum(m.x1[h,i] for i in m.I) <= 100000000000*m.y1[h] #Big M constraint
m.con5 = Constraint(m.H, rule = capacity_const)


opt = SolverFactory('gurobi')
results = opt.solve(m, tee = True)

print(results)


!cat results.yml


Set parameter TokenServer to value "coe-vtls1.engr.tamu.edu"
Read LP format model from file C:\Users\BETSIE~1\AppData\Local\Temp\tmp4afk7t73.pyomo.lp
Reading time = 0.00 seconds
x1: 15 rows, 22 columns, 72 nonzeros
Gurobi Optimizer version 10.0.0 build v10.0.0rc2 (win64)

CPU model: Intel(R) Core(TM) i7-4790 CPU @ 3.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 15 rows, 22 columns and 72 nonzeros
Model fingerprint: 0xed3ad548
Variable types: 17 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+11]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+03, 7e+03]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Presolve removed 7 rows and 2 columns
Presolve time: 0.00s
Presolved: 8 rows, 20 columns, 35 nonzeros
Variable types: 15 continuous, 5 integer (5 binary)
Foun

'cat' is not recognized as an internal or external command,
operable program or batch file.


In [15]:
results.pprint

<bound method MapContainer.pprint of {'Problem': [{'Name': 'x1', 'Lower bound': 418000.0, 'Upper bound': 418000.0, 'Number of objectives': 1, 'Number of constraints': 15, 'Number of variables': 22, 'Number of binary variables': 5, 'Number of integer variables': 5, 'Number of continuous variables': 17, 'Number of nonzeros': 72, 'Sense': 'minimize'}], 'Solver': [{'Status': 'ok', 'Return code': '0', 'Message': 'Model was solved to optimality (subject to tolerances), and an optimal solution is available.', 'Termination condition': 'optimal', 'Termination message': 'Model was solved to optimality (subject to tolerances), and an optimal solution is available.', 'Wall time': '0.006999969482421875', 'Error rc': 0, 'Time': 0.23435759544372559}], 'Solution': [OrderedDict([('number of solutions', 0), ('number of solutions displayed', 0)])]}>

In [16]:
m.x1.pprint()

x1 : Size=15, Index=x1_index
    Key                       : Lower : Value  : Upper : Fixed : Stale : Domain
          ('Bryan', 'Austin') :     0 : 4000.0 :  None : False : False : NonNegativeReals
          ('Bryan', 'Dallas') :     0 : 1000.0 :  None : False : False : NonNegativeReals
         ('Bryan', 'Houston') :     0 : 1000.0 :  None : False : False : NonNegativeReals
         ('Conroe', 'Austin') :     0 :    0.0 :  None : False : False : NonNegativeReals
         ('Conroe', 'Dallas') :     0 :    0.0 :  None : False : False : NonNegativeReals
        ('Conroe', 'Houston') :     0 : 2000.0 :  None : False : False : NonNegativeReals
      ('Hempstead', 'Austin') :     0 :    0.0 :  None : False : False : NonNegativeReals
      ('Hempstead', 'Dallas') :     0 :    0.0 :  None : False : False : NonNegativeReals
     ('Hempstead', 'Houston') :     0 :    0.0 :  None : False : False : NonNegativeReals
     ('Montgomery', 'Austin') :     0 :    0.0 :  None : False : False : NonNegat

In [17]:
m.total_cost.pprint()

total_cost : Size=1, Index=None, Active=True
    Key  : Active : Sense    : Expression
    None :   True : minimize : a + b
